# S7 Colour v8 — GPU development qualification

This notebook trains on the frozen **development** archive, selects architecture/epoch on validation only, and calibrates only the selected checkpoint. It never creates or reads the new external final and never enables SaaS integration.

**Recovery update:** it uses the authenticated Google Drive API instead of `drive.mount()`, which avoids the DriveFS mount timeout while keeping every source and result private.

## 1. Authenticate the private Drive API

Google may ask for authorization once. Do not paste credentials, tokens, or GitHub secrets into the notebook.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from google.auth import default
from pathlib import Path
import hashlib, json, mimetypes, os, shutil, subprocess, sys

credentials, _ = default()
drive_service = build('drive', 'v3', credentials=credentials, cache_discovery=False)

def drive_download(file_id, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    metadata = drive_service.files().get(
        fileId=file_id,
        fields='id,name,size,md5Checksum',
        supportsAllDrives=True,
    ).execute(num_retries=5)
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with destination.open('wb') as stream:
        transfer = MediaIoBaseDownload(stream, request, chunksize=16 * 1024 * 1024)
        done = False
        while not done:
            status, done = transfer.next_chunk(num_retries=5)
            if status:
                print(f"download {metadata['name']}: {status.progress():.0%}")
    if metadata.get('size'):
        assert destination.stat().st_size == int(metadata['size'])
    return destination

def drive_create_folder(name, parent_id):
    result = drive_service.files().create(
        body={
            'name': name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_id],
        },
        fields='id,name',
        supportsAllDrives=True,
    ).execute(num_retries=5)
    return result['id']

def drive_upload_file(local_path, parent_id):
    local_path = Path(local_path)
    media = MediaFileUpload(
        str(local_path),
        mimetype=mimetypes.guess_type(local_path.name)[0] or 'application/octet-stream',
        resumable=True,
        chunksize=16 * 1024 * 1024,
    )
    result = drive_service.files().create(
        body={'name': local_path.name, 'parents': [parent_id]},
        media_body=media,
        fields='id,name,size',
        supportsAllDrives=True,
    ).execute(num_retries=5)
    print({'uploaded': result['name'], 'bytes': result.get('size'), 'file_id': result['id']})
    return result['id']

def drive_upload_tree(local_dir, parent_id):
    uploaded = {}
    for child in sorted(Path(local_dir).iterdir()):
        if child.is_dir():
            folder_id = drive_create_folder(child.name, parent_id)
            uploaded[child.name] = {'folder_id': folder_id, 'children': drive_upload_tree(child, folder_id)}
        else:
            uploaded[child.name] = drive_upload_file(child, parent_id)
    return uploaded

print('Private Drive API authenticated.')

In [ ]:
from datetime import datetime, timezone

GITHUB_REPOSITORY = 'getibplay-cmyk/pfe'
GITHUB_BRANCH = 'codex/s7-color-v8-colab-gpu'
GITHUB_SOURCE_COMMIT = '92b747f125a3111fc2b0aa0462b651bc91744dd6'

SOURCE_BUNDLE_FILE_ID = '1eJuytbkZG38nqWBRQNVRCJ5TStIniPNg'
EXPECTED_SOURCE_BUNDLE_SHA256 = '66fa0da68fdce41b8a763c9cb1de960e2116cb5b5ee2de832cc6beb949594e50'
DRIVE_DATA_FOLDER_ID = '1yRFkIvDrylu3Xi1jWJeQrt1QngQBa5Pz'
DRIVE_MODELS_FOLDER_ID = '1RpNjvNn27VOUPpfvHyHYfUpR6nGedF29'
EXPECTED_ARCHIVE_SHA256 = 'ceb971e7af86194a56d0c33a4d10356c7174d911a33714d7e7336c27e164f62b'

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_RUN_FOLDER_ID = drive_create_folder(RUN_ID, DRIVE_MODELS_FOLDER_ID)
LOCAL_REPO = Path('/content/pfe')
LOCAL_DATA = Path('/content/s7_color_v8_development')
LOCAL_RUN = Path('/content/s7_color_v8_run')

for local_path in (LOCAL_REPO, LOCAL_DATA, LOCAL_RUN):
    if local_path.exists():
        shutil.rmtree(local_path)
LOCAL_RUN.mkdir(parents=True, exist_ok=False)

print({
    'run_id': RUN_ID,
    'private_drive_run': f'https://drive.google.com/drive/folders/{DRIVE_RUN_FOLDER_ID}',
})

## 2. CUDA and immutable source bundle

If CUDA fails, choose **Runtime → Change runtime type → GPU**, then restart from the beginning. Colab GPU type and availability can vary.

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA is mandatory for the real run'
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_bundle = drive_download(SOURCE_BUNDLE_FILE_ID, '/content/S7_COLOR_V8_SOURCE_BUNDLE.zip')
assert sha256_file(source_bundle) == EXPECTED_SOURCE_BUNDLE_SHA256
subprocess.run(['unzip', '-q', str(source_bundle), '-d', '/content'], check=True)
SCRIPT_DIR = LOCAL_REPO / 'scripts/intelligence/color_v8'
assert (SCRIPT_DIR / 'train_color_v8.py').is_file()
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(SCRIPT_DIR / 'requirements-color-v8.txt'),
], check=True)
print({
    'github_repository': GITHUB_REPOSITORY,
    'github_branch': GITHUB_BRANCH,
    'source_commit': GITHUB_SOURCE_COMMIT,
    'source_bundle_sha256': sha256_file(source_bundle),
})

In [ ]:
DATA_FILE_IDS = {
    'S7_COLOR_V8_DEVELOPMENT_REGISTRY.json': '1n3MzlqsmX5kSdlCqjoBvkQAZup6qkm40',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.multipart.json': '1CEA34pUvQ1aE8HO4_7Ik3q2kZxNwN0BJ',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part000': '1H_1oJyXD5CBwi_6lnV-354YWMScNCbAb',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part001': '165GaIIarYR1FUjvRq1Ndxx8afUOkuuyq',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part002': '11syKntE-vkwL3xhzqnBAoikZ4Fn9r62V',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part003': '1W1QvtuOTGjp7OAPb2mfikwpDjZJgGgFS',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part004': '1uSrVjsFJwUbsl4S5is7lJUwUBVwF1DlF',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part005': '1JDSCDJ3sUkSw-CuFFR08jJsOLqvR4iso',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part006': '1mIRZiaxcuUBQaaJX4DFQQi-pm9xyoGkc',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part007': '1MXWptuYblpY1cvKQBXPrRbZPvLco3fYz',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part008': '15932omJIxO87vcM9LteZE7vGAUDqS9k2',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part009': '1jiZmditY3xNTrWLIHNzGt0dPvS4mUBVR',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part010': '1OPVQGmU9TH-kpHCd8W8GteUfGTb_evqz',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part011': '1K2tyv1yxhU6KWVY9WIMV8oreYaJsrs0W',
    'S7_COLOR_V8_DEVELOPMENT_DATA.zip.part012': '1MCFb3W0pFFS61_Qib5XoXg1cMMLyosfd',
}

registry_path = drive_download(
    DATA_FILE_IDS['S7_COLOR_V8_DEVELOPMENT_REGISTRY.json'],
    '/content/S7_COLOR_V8_DEVELOPMENT_REGISTRY.json',
)
multipart_path = drive_download(
    DATA_FILE_IDS['S7_COLOR_V8_DEVELOPMENT_DATA.zip.multipart.json'],
    '/content/S7_COLOR_V8_DEVELOPMENT_DATA.zip.multipart.json',
)
registry = json.loads(registry_path.read_text())
multipart = json.loads(multipart_path.read_text())
assert registry['status'] == 'DEVELOPMENT_ONLY_NEW_FINAL_NOT_CREATED'
assert registry['artifacts']['S7_COLOR_V8_DEVELOPMENT_DATA.zip']['sha256'] == EXPECTED_ARCHIVE_SHA256
assert multipart['archive']['sha256'] == EXPECTED_ARCHIVE_SHA256

local_archive = Path('/content/S7_COLOR_V8_DEVELOPMENT_DATA.zip')
if local_archive.exists():
    local_archive.unlink()
with local_archive.open('wb') as destination:
    for expected_part in multipart['parts']:
        part_name = expected_part['name']
        assert part_name in DATA_FILE_IDS, f'Unregistered Drive part: {part_name}'
        part_path = drive_download(DATA_FILE_IDS[part_name], Path('/content') / part_name)
        assert part_path.stat().st_size == expected_part['bytes']
        assert sha256_file(part_path) == expected_part['sha256']
        with part_path.open('rb') as source:
            shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
        part_path.unlink()

assert local_archive.stat().st_size == multipart['archive']['bytes']
assert sha256_file(local_archive) == EXPECTED_ARCHIVE_SHA256
LOCAL_DATA.mkdir(parents=True, exist_ok=False)
subprocess.run(['unzip', '-q', str(local_archive), '-d', str(LOCAL_DATA)], check=True)
manifest = LOCAL_DATA / 'S7_COLOR_V8_DEVELOPMENT_MANIFEST.csv'
assert sha256_file(manifest) == registry['artifacts'][manifest.name]['sha256']
print({
    'rows': registry['retained_rows'],
    'archive_sha256': sha256_file(local_archive),
    'manifest_sha256': sha256_file(manifest),
    'final_status': registry['new_final']['status'],
})

## 3. Candidate training — validation only

Each completed candidate is immediately checkpointed through the private Drive API. The training script structurally refuses calibration and final access.

In [ ]:
candidate_plan = [
    ('mobilenet_v3_large', 96),
    ('efficientnet_v2_s', 48),
    ('convnext_tiny', 64),
]
drive_candidates_id = drive_create_folder('candidates', DRIVE_RUN_FOLDER_ID)
candidate_reports = []

for model_name, batch_size in candidate_plan:
    candidate_dir = LOCAL_RUN / 'candidates' / model_name
    command = [
        sys.executable, str(SCRIPT_DIR / 'train_color_v8.py'),
        '--dataset-root', str(LOCAL_DATA),
        '--manifest', str(manifest),
        '--output-dir', str(candidate_dir),
        '--model-name', model_name,
        '--epochs', '18', '--patience', '5',
        '--batch-size', str(batch_size), '--workers', '2',
    ]
    subprocess.run(command, check=True)
    report = next(candidate_dir.glob('*_CANDIDATE_REPORT.json'))
    candidate_reports.append(report)
    drive_model_id = drive_create_folder(model_name, drive_candidates_id)
    drive_upload_tree(candidate_dir, drive_model_id)
    print({'checkpointed': model_name, 'private_drive_folder_id': drive_model_id})

## 4. Deterministic selection, then calibration of one candidate

In [ ]:
selection_dir = LOCAL_RUN / 'selection'
subprocess.run([
    sys.executable, str(SCRIPT_DIR / 'select_color_v8_candidate.py'),
    '--reports', *[str(path) for path in candidate_reports],
    '--output-dir', str(selection_dir),
], check=True)
drive_selection_id = drive_create_folder('selection', DRIVE_RUN_FOLDER_ID)
drive_upload_tree(selection_dir, drive_selection_id)
selection_ledger = json.loads((selection_dir / 'S7_COLOR_V8_SELECTION_LEDGER.json').read_text())
print({
    'selected': selection_ledger['selected']['candidate'],
    'calibration_loaded_during_selection': selection_ledger['calibration_images_loaded'],
})

In [ ]:
qualification_dir = LOCAL_RUN / 'qualification'
subprocess.run([
    sys.executable, str(SCRIPT_DIR / 'qualify_color_v8_development.py'),
    '--dataset-root', str(LOCAL_DATA),
    '--manifest', str(manifest),
    '--selection-dir', str(selection_dir),
    '--output-dir', str(qualification_dir),
    '--batch-size', '128', '--workers', '2',
], check=True)

drive_qualification_id = drive_create_folder('qualification', DRIVE_RUN_FOLDER_ID)
drive_upload_tree(qualification_dir, drive_qualification_id)
qualification_report = json.loads(
    (qualification_dir / 'S7_COLOR_V8_DEVELOPMENT_QUALIFICATION_REPORT.json').read_text()
)
run_ledger = {
    'schema_version': '8.0.0',
    'run_id': RUN_ID,
    'github_repository': GITHUB_REPOSITORY,
    'github_branch': GITHUB_BRANCH,
    'source_commit': GITHUB_SOURCE_COMMIT,
    'source_bundle_sha256': EXPECTED_SOURCE_BUNDLE_SHA256,
    'gpu': torch.cuda.get_device_name(0),
    'development_archive_sha256': EXPECTED_ARCHIVE_SHA256,
    'selected_candidate': selection_ledger['selected']['candidate'],
    'development_gate_passed': qualification_report['decisions']['development_gate_passed'],
    'new_external_final_authorized': qualification_report['decisions']['new_external_final_authorized'],
    'new_external_final_executed': False,
    'saas_integration_authorized': False,
}
ledger_path = LOCAL_RUN / 'S7_COLOR_V8_COLAB_RUN_LEDGER.json'
ledger_path.write_text(json.dumps(run_ledger, indent=2, sort_keys=True) + '\n')
drive_upload_file(ledger_path, DRIVE_RUN_FOLDER_ID)
print(json.dumps(run_ledger, indent=2))
print({'private_drive_run': f'https://drive.google.com/drive/folders/{DRIVE_RUN_FOLDER_ID}'})

## STOP

This notebook intentionally stops here. Even when the development gate passes, do **not** improvise a final run. First freeze a newly sourced, prediction-blind, independently licensed final with `freeze_color_v8_external_final.py`; then execute `evaluate_color_v8_external_final_once.py` exactly once. ONNX export and SaaS integration remain forbidden until that report passes.

## Runtime recovery — completed gate-passing candidate

Colab disabled the first T4 runtime after MobileNetV3-Large had completed and been checkpointed privately. These recovery cells continue the **same audited run** from that immutable checkpoint. They do not retrain it, do not claim a three-architecture comparison, and do not access an external final.

In [ ]:
# Run cell 2 (Drive authentication helpers) before this recovery cell.
from datetime import datetime, timezone

GITHUB_REPOSITORY = 'getibplay-cmyk/pfe'
GITHUB_BRANCH = 'codex/s7-color-v8-colab-gpu'
GITHUB_SOURCE_COMMIT = '92b747f125a3111fc2b0aa0462b651bc91744dd6'
SOURCE_BUNDLE_FILE_ID = '1eJuytbkZG38nqWBRQNVRCJ5TStIniPNg'
EXPECTED_SOURCE_BUNDLE_SHA256 = '66fa0da68fdce41b8a763c9cb1de960e2116cb5b5ee2de832cc6beb949594e50'
EXPECTED_ARCHIVE_SHA256 = 'ceb971e7af86194a56d0c33a4d10356c7174d911a33714d7e7336c27e164f62b'
RUN_ID = '20260822T184459Z'
DRIVE_RUN_FOLDER_ID = '1NIXAB0YK7GGpHTUZf84dwgc8MK-zY8L1'
MOBILENET_REPORT_FILE_ID = '1B5UEdGa0JwoKqlyx0qPr26-TzRzYp3nP'
MOBILENET_STATE_FILE_ID = '1KmcL2J7Fa8MmOgX-TeOG30AZ8owmR2LN'

LOCAL_REPO = Path('/content/pfe')
LOCAL_DATA = Path('/content/s7_color_v8_development')
LOCAL_RUN = Path('/content/s7_color_v8_run')
for local_path in (LOCAL_REPO, LOCAL_DATA, LOCAL_RUN):
    if local_path.exists():
        shutil.rmtree(local_path)
LOCAL_RUN.mkdir(parents=True, exist_ok=False)

subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA is mandatory for the recovery qualification'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_bundle = drive_download(SOURCE_BUNDLE_FILE_ID, '/content/S7_COLOR_V8_SOURCE_BUNDLE.zip')
assert sha256_file(source_bundle) == EXPECTED_SOURCE_BUNDLE_SHA256
subprocess.run(['unzip', '-q', str(source_bundle), '-d', '/content'], check=True)
SCRIPT_DIR = LOCAL_REPO / 'scripts/intelligence/color_v8'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(SCRIPT_DIR / 'requirements-color-v8.txt'),
], check=True)

candidate_dir = LOCAL_RUN / 'candidates' / 'mobilenet_v3_large'
candidate_dir.mkdir(parents=True, exist_ok=False)
candidate_report_path = drive_download(
    MOBILENET_REPORT_FILE_ID,
    candidate_dir / 'S7_COLOR_V8_MOBILENET_V3_LARGE_CANDIDATE_REPORT.json',
)
candidate_state_path = drive_download(
    MOBILENET_STATE_FILE_ID,
    candidate_dir / 'S7_COLOR_V8_MOBILENET_V3_LARGE_CANDIDATE_STATE.pt',
)
candidate_report = json.loads(candidate_report_path.read_text())
assert candidate_report['decisions']['candidate_validation_gate_passed'] is True
assert candidate_report['selection_protocol']['calibration_images_loaded'] is False
assert candidate_report['selection_protocol']['final_images_loaded'] is False
state_metadata = candidate_report['artifacts'][candidate_state_path.name]
assert candidate_state_path.stat().st_size == state_metadata['bytes']
assert sha256_file(candidate_state_path) == state_metadata['sha256']
print({
    'recovery': 'same_private_run',
    'gpu': torch.cuda.get_device_name(0),
    'candidate': candidate_report['candidate'],
    'candidate_gate_passed': True,
    'candidate_state_sha256': state_metadata['sha256'],
    'external_final_loaded': False,
})

Run the original dataset-transfer cell (cell 7) next. It reconstructs and verifies the immutable development archive locally, following Colab's recommended archive-first Drive pattern. Then run the cell below.

In [ ]:
candidate_reports = [candidate_report_path]
selection_dir = LOCAL_RUN / 'selection'
subprocess.run([
    sys.executable, str(SCRIPT_DIR / 'select_color_v8_candidate.py'),
    '--reports', str(candidate_report_path),
    '--output-dir', str(selection_dir),
], check=True)
selection_ledger = json.loads((selection_dir / 'S7_COLOR_V8_SELECTION_LEDGER.json').read_text())
assert selection_ledger['selected']['candidate'] == 'mobilenet_v3_large'
assert selection_ledger['calibration_images_loaded'] is False
assert selection_ledger['final_images_loaded'] is False
drive_selection_id = drive_create_folder('selection', DRIVE_RUN_FOLDER_ID)
drive_upload_tree(selection_dir, drive_selection_id)

qualification_dir = LOCAL_RUN / 'qualification'
subprocess.run([
    sys.executable, str(SCRIPT_DIR / 'qualify_color_v8_development.py'),
    '--dataset-root', str(LOCAL_DATA),
    '--manifest', str(manifest),
    '--selection-dir', str(selection_dir),
    '--output-dir', str(qualification_dir),
    '--batch-size', '128', '--workers', '2',
], check=True)
qualification_report = json.loads(
    (qualification_dir / 'S7_COLOR_V8_DEVELOPMENT_QUALIFICATION_REPORT.json').read_text()
)
drive_qualification_id = drive_create_folder('qualification', DRIVE_RUN_FOLDER_ID)
drive_upload_tree(qualification_dir, drive_qualification_id)

recovery_audit = {
    'schema_version': '8.0.0',
    'created_at_utc': datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z'),
    'run_id': RUN_ID,
    'incident': 'COLAB_RUNTIME_DISABLED_DURING_SECOND_CANDIDATE',
    'completed_candidate': 'mobilenet_v3_large',
    'completed_candidate_validation_gate_passed': True,
    'incomplete_candidates_not_used': ['efficientnet_v2_s', 'convnext_tiny'],
    'selection_scope': 'sole_completed_gate_passing_candidate',
    'claim_of_global_architecture_optimality': False,
    'external_final_loaded': False,
    'external_final_executed': False,
    'saas_integration_authorized': False,
}
recovery_path = LOCAL_RUN / 'S7_COLOR_V8_RUNTIME_RECOVERY_AUDIT.json'
recovery_path.write_text(json.dumps(recovery_audit, indent=2, sort_keys=True) + '\n')
drive_upload_file(recovery_path, DRIVE_RUN_FOLDER_ID)

run_ledger = {
    'schema_version': '8.0.0',
    'run_id': RUN_ID,
    'github_repository': GITHUB_REPOSITORY,
    'github_branch': GITHUB_BRANCH,
    'source_commit': GITHUB_SOURCE_COMMIT,
    'source_bundle_sha256': EXPECTED_SOURCE_BUNDLE_SHA256,
    'gpu': torch.cuda.get_device_name(0),
    'development_archive_sha256': EXPECTED_ARCHIVE_SHA256,
    'selected_candidate': selection_ledger['selected']['candidate'],
    'selection_scope': recovery_audit['selection_scope'],
    'development_gate_passed': qualification_report['decisions']['development_gate_passed'],
    'new_external_final_authorized': qualification_report['decisions']['new_external_final_authorized'],
    'new_external_final_executed': False,
    'saas_integration_authorized': False,
}
ledger_path = LOCAL_RUN / 'S7_COLOR_V8_COLAB_RUN_LEDGER.json'
ledger_path.write_text(json.dumps(run_ledger, indent=2, sort_keys=True) + '\n')
drive_upload_file(ledger_path, DRIVE_RUN_FOLDER_ID)
print(json.dumps({
    'run_ledger': run_ledger,
    'development_gate_checks': qualification_report['gate_checks'],
    'temperature': qualification_report['calibration']['temperature'],
    'private_drive_run': f'https://drive.google.com/drive/folders/{DRIVE_RUN_FOLDER_ID}',
}, indent=2))

## Calibrated abstention recovery

The first qualification preserved below failed only the calibration reject checks: 12/232 aggregate false accepts and 11/150 on one source at threshold 0.90. The recovery source calibrates the **minimum feasible threshold from 0.90 to 0.99 on calibration only**. Candidate weights, validation-only selection, temperature fitting, and the untouched external-final boundary remain unchanged.

In [ ]:
CALIBRATED_SOURCE_BUNDLE_FILE_ID = '1-tp-7unQhLsvytFl2iBcMpW4jypqAtw3'
EXPECTED_CALIBRATED_SOURCE_SHA256 = '218acc289a330f500fbd1d911390f2f04c2c079d8830c934856cfe743d8367fd'
QUALIFICATION_SOURCE_COMMIT = 'c8a5210689410f1155f52439ac6885e998624232'

calibrated_bundle = drive_download(
    CALIBRATED_SOURCE_BUNDLE_FILE_ID,
    '/content/S7_COLOR_V8_SOURCE_BUNDLE_CALIBRATED_THRESHOLD.zip',
)
assert sha256_file(calibrated_bundle) == EXPECTED_CALIBRATED_SOURCE_SHA256
CALIBRATED_SOURCE_ROOT = Path('/content/s7_color_v8_calibrated_source')
if CALIBRATED_SOURCE_ROOT.exists():
    shutil.rmtree(CALIBRATED_SOURCE_ROOT)
CALIBRATED_SOURCE_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(['unzip', '-q', str(calibrated_bundle), '-d', str(CALIBRATED_SOURCE_ROOT)], check=True)
CALIBRATED_SCRIPT_DIR = CALIBRATED_SOURCE_ROOT / 'pfe/scripts/intelligence/color_v8'
subprocess.run([
    sys.executable, '-m', 'py_compile',
    str(CALIBRATED_SCRIPT_DIR / 'train_color_v8.py'),
    str(CALIBRATED_SCRIPT_DIR / 'qualify_color_v8_development.py'),
    str(CALIBRATED_SCRIPT_DIR / 'evaluate_color_v8_external_final_once.py'),
], check=True)

prior_qualification_report = json.loads(
    (qualification_dir / 'S7_COLOR_V8_DEVELOPMENT_QUALIFICATION_REPORT.json').read_text()
)
assert prior_qualification_report['decisions']['development_gate_passed'] is False
assert prior_qualification_report['gate_checks']['calibration_reject'] is False
assert prior_qualification_report['gate_checks']['calibration_source_reject'] is False

calibrated_qualification_dir = LOCAL_RUN / 'qualification_calibrated_threshold'
subprocess.run([
    sys.executable, str(CALIBRATED_SCRIPT_DIR / 'qualify_color_v8_development.py'),
    '--dataset-root', str(LOCAL_DATA),
    '--manifest', str(manifest),
    '--selection-dir', str(selection_dir),
    '--output-dir', str(calibrated_qualification_dir),
    '--batch-size', '128', '--workers', '2',
], check=True)
calibrated_report_path = calibrated_qualification_dir / 'S7_COLOR_V8_DEVELOPMENT_QUALIFICATION_REPORT.json'
calibrated_report = json.loads(calibrated_report_path.read_text())
assert calibrated_report['decisions']['development_gate_passed'] is True
assert calibrated_report['decisions']['new_external_final_authorized'] is True
assert calibrated_report['decisions']['new_external_final_executed'] is False
assert calibrated_report['decisions']['saas_integration_authorized'] is False
confidence_threshold = calibrated_report['calibration']['confidence_threshold']
assert 0.90 <= confidence_threshold <= 0.99

drive_calibrated_qualification_id = drive_create_folder(
    'qualification_calibrated_threshold',
    DRIVE_RUN_FOLDER_ID,
)
drive_upload_tree(calibrated_qualification_dir, drive_calibrated_qualification_id)

calibration_recovery_audit = {
    'schema_version': '8.0.0',
    'created_at_utc': datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z'),
    'run_id': RUN_ID,
    'candidate_weights_changed': False,
    'selection_ledger_changed': False,
    'previous_development_gate_passed': False,
    'previous_failed_checks': ['calibration_reject', 'calibration_source_reject'],
    'recovery_method': 'minimum_feasible_threshold_on_fixed_0_90_to_0_99_calibration_grid',
    'selected_confidence_threshold': confidence_threshold,
    'candidate_training_source_commit': GITHUB_SOURCE_COMMIT,
    'qualification_source_commit': QUALIFICATION_SOURCE_COMMIT,
    'qualification_source_bundle_sha256': EXPECTED_CALIBRATED_SOURCE_SHA256,
    'candidate_state_sha256': state_metadata['sha256'],
    'external_final_loaded': False,
    'external_final_executed': False,
    'saas_integration_authorized': False,
}
calibration_audit_path = LOCAL_RUN / 'S7_COLOR_V8_CALIBRATED_THRESHOLD_RECOVERY_AUDIT.json'
calibration_audit_path.write_text(json.dumps(calibration_recovery_audit, indent=2, sort_keys=True) + '\n')
drive_upload_file(calibration_audit_path, DRIVE_RUN_FOLDER_ID)

final_development_ledger = {
    'schema_version': '8.0.0',
    'run_id': RUN_ID,
    'github_repository': GITHUB_REPOSITORY,
    'github_branch': GITHUB_BRANCH,
    'candidate_training_source_commit': GITHUB_SOURCE_COMMIT,
    'qualification_source_commit': QUALIFICATION_SOURCE_COMMIT,
    'candidate_source_bundle_sha256': EXPECTED_SOURCE_BUNDLE_SHA256,
    'qualification_source_bundle_sha256': EXPECTED_CALIBRATED_SOURCE_SHA256,
    'gpu': torch.cuda.get_device_name(0),
    'development_archive_sha256': EXPECTED_ARCHIVE_SHA256,
    'selected_candidate': selection_ledger['selected']['candidate'],
    'selection_scope': recovery_audit['selection_scope'],
    'confidence_threshold': confidence_threshold,
    'development_gate_passed': True,
    'new_external_final_authorized': True,
    'new_external_final_executed': False,
    'saas_integration_authorized': False,
}
final_ledger_path = LOCAL_RUN / 'S7_COLOR_V8_COLAB_RUN_LEDGER_CALIBRATED_THRESHOLD.json'
final_ledger_path.write_text(json.dumps(final_development_ledger, indent=2, sort_keys=True) + '\n')
drive_upload_file(final_ledger_path, DRIVE_RUN_FOLDER_ID)
print(json.dumps({
    'run_ledger': final_development_ledger,
    'development_gate_checks': calibrated_report['gate_checks'],
    'temperature': calibrated_report['calibration']['temperature'],
    'confidence_threshold': confidence_threshold,
    'validation': {
        'accepted_precision': calibrated_report['metrics']['validation']['accepted_precision_at_threshold'],
        'coverage': calibrated_report['metrics']['validation']['coverage_at_threshold'],
        'reject_false_acceptance_rate': calibrated_report['metrics']['validation_reject']['false_acceptance_rate_at_threshold'],
    },
    'calibration': {
        'accepted_precision': calibrated_report['metrics']['calibration']['accepted_precision_at_threshold'],
        'coverage': calibrated_report['metrics']['calibration']['coverage_at_threshold'],
        'reject_false_acceptance_rate': calibrated_report['metrics']['calibration_reject']['false_acceptance_rate_at_threshold'],
    },
    'private_drive_run': f'https://drive.google.com/drive/folders/{DRIVE_RUN_FOLDER_ID}',
}, indent=2))